# Gemma 4 AML — Inference Only (vast.ai)

Downloads a pre-trained GGUF from HuggingFace Hub, registers it with Ollama, and starts serving.
No training required.

**Pipeline:**
1. Download GGUF from HF Hub
2. Install Ollama
3. Write Modelfile (Gemma 4 chat template)
4. Register model + start server
5. Smoke test

**Recommended vast.ai instance:** RTX 3090 24GB, **30GB disk** minimum (GGUF is ~8 GB q8_0)

## Cell 1 — Configuration

In [ ]:
# ── Which model to pull ────────────────────────────────────────────────────
HF_REPO        = "speri420/aria-v2"
HF_FILENAME    = "aria-v2-q8.gguf"
HF_TOKEN       = ""                       # leave blank — repo is public

GGUF_PATH      = f"/workspace/{HF_FILENAME}"
MODELFILE_PATH = "/tmp/Modelfile.aml"
OLLAMA_MODEL   = "aria-v2"

print(f"Model repo : {HF_REPO}")
print(f"GGUF path  : {GGUF_PATH}")
print(f"Ollama name: {OLLAMA_MODEL}")

## Cell 2 — Download GGUF from HuggingFace Hub

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

from huggingface_hub import hf_hub_download
from pathlib import Path

gguf = Path(GGUF_PATH)

if gguf.exists() and gguf.stat().st_size > 1_000_000_000:
    print(f"GGUF already present: {gguf}  ({gguf.stat().st_size / 1e9:.1f} GB) — skipping download.")
else:
    print(f"Downloading {HF_REPO}/{HF_FILENAME} → {GGUF_PATH} ...")
    kwargs = dict(
        repo_id=HF_REPO,
        filename=HF_FILENAME,
        local_dir="/workspace",
    )
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    hf_hub_download(**kwargs)
    print(f"Download complete: {gguf.stat().st_size / 1e9:.1f} GB")

## Cell 3 — Install Ollama

In [ ]:
import shutil, subprocess

if shutil.which("ollama") is None:
    print("Installing Ollama ...")
    result = subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True, capture_output=True, text=True
    )
    print(result.stdout[-2000:] if result.stdout else "(no stdout)")
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
    else:
        print("Ollama installed.")
else:
    v = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    print(f"Ollama already installed: {v.stdout.strip()}")

## Cell 4 — Write Modelfile

Matches Cell 11 in the training notebook exactly — Gemma 4 format (`<|turn>user` / `<turn|>`).
`num_ctx 8192` and two stop tokens (`<turn|>`, `<eos>`) match training configuration.

In [ ]:
from pathlib import Path

SYSTEM_PROMPT = (
    'You are an AML (Anti-Money Laundering) analytics AI assistant. '
    'You analyze false positive/false negative trade-offs in AML alert thresholds, '
    'perform customer behavioral segmentation, and interpret clustering results. '
    'Use the available tools to retrieve data, then provide clear, analytical insights. '
    'IMPORTANT: You MUST respond entirely in English. '
    'Be concise and reference specific numbers when interpreting results.'
)

lines = [
    f'FROM {GGUF_PATH}',
    '',
    'PARAMETER num_ctx 8192',
    'PARAMETER temperature 0.1',
    'PARAMETER top_p 0.9',
    'PARAMETER stop <turn|>',
    'PARAMETER stop <eos>',
    '',
    'TEMPLATE """',
    '{{ if .System }}<|turn>user',
    '{{ .System }}<turn|>',
    '{{ end }}',
    '{{ range .Messages }}',
    '{{ if eq .Role "user" }}<|turn>user',
    '{{ .Content }}<turn|>',
    '<|turn>model',
    '{{ else if eq .Role "assistant" }}{{ .Content }}<turn|>',
    '{{ else if eq .Role "tool" }}<|turn>tool',
    '{{ .Content }}<turn|>',
    '<|turn>model',
    '{{ end }}',
    '{{ end }}',
    '"""',
    '',
    f'SYSTEM "{SYSTEM_PROMPT}"',
]

modelfile = '\n'.join(lines)
Path(MODELFILE_PATH).write_text(modelfile, encoding='utf-8')
print(f'Modelfile written to {MODELFILE_PATH}')
print('--- Preview ---')
print(modelfile)

## Cell 5 — Start Ollama Server

⚠️ If the process dies after notebook close, restart from terminal:
```bash
OLLAMA_HOST=0.0.0.0:11434 nohup ollama serve > /tmp/ollama.log 2>&1 &
```

In [ ]:
import os, subprocess, time

subprocess.run("pkill -f 'ollama serve' || true", shell=True)
time.sleep(2)

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"

subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("Waiting for Ollama to start ...")
time.sleep(6)

check = subprocess.run(["curl", "-s", "http://localhost:11434/api/tags"],
                       capture_output=True, text=True)
if check.returncode == 0 and "models" in check.stdout:
    print("Ollama is up.")
else:
    print("Ollama may still be starting — wait a few seconds and re-run, or start from terminal.")

## Cell 6 — Register Model with Ollama

In [ ]:
import subprocess

print(f"Registering '{OLLAMA_MODEL}' from {MODELFILE_PATH} ...")
result = subprocess.run(
    ["ollama", "create", OLLAMA_MODEL, "-f", MODELFILE_PATH],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)
else:
    print(f"Model '{OLLAMA_MODEL}' registered successfully.")

lst = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(lst.stdout)

## Cell 7 — Smoke Test

Sends a plain chat request. The model generates tool calls in `tool_code` format:
```
tool_code print(threshold_tuning(customer_type='Business', threshold_column='trxn_amt_monthly'))
```
The app's fallback parsers (Formats 1–8 in `base_agent.py`) extract the call from the content text.

In [ ]:
import urllib.request, json, re

payload = json.dumps({
    "model": OLLAMA_MODEL,
    "messages": [
        {"role": "user", "content": "Show FP/FN trade-off for Business customers by monthly transaction amount"}
    ],
    "stream": False,
}).encode()

req = urllib.request.Request(
    "http://localhost:11434/api/chat",
    data=payload,
    headers={"Content-Type": "application/json"}
)

with urllib.request.urlopen(req, timeout=120) as resp:
    result = json.loads(resp.read())

content = result["message"]["content"]
print("Response:")
print(content[:800])

# Check for tool_code format: tool_code print(function_name(...))
if re.search(r'tool_code\s+print\s*\(', content):
    m = re.search(r'tool_code\s+print\s*\((\w+)', content)
    fn = m.group(1) if m else "unknown"
    print(f"\n✓ tool_code call detected: {fn}")
    print("SMOKE TEST PASSED")
else:
    print("\n⚠ No tool_code pattern found — check response above.")

## Cell 8 — Regression Test (key V15 prompts)

Tests SAR worklist, network graph, and help routing.

In [ ]:
import urllib.request, json, re

SYSTEM = (
    'You are a FRAML (Fraud + AML) analytics AI assistant. '
    'Use the available tools to retrieve data, then provide clear, analytical insights. '
    'Be concise and reference specific numbers when interpreting results.'
)

# (prompt, expected_tool_fn_name or None, keyword_if_text)
test_cases = [
    ('Show FP/FN trade-off for Business customers',          'threshold_tuning',     None),
    ('Run SAR backtest for Activity Deviation ACH rule',     'rule_sar_backtest',     None),
    ('Segment Business customers into clusters',             'ss_cluster_analysis',   None),
    ('Show me all AML rules',                               'list_rules',            None),
    ('What can you help me with?',                           None,                   'Threshold'),
]

def ask(user_msg):
    payload = json.dumps({
        'model': OLLAMA_MODEL,
        'messages': [
            {'role': 'system', 'content': SYSTEM},
            {'role': 'user',   'content': user_msg},
        ],
        'stream': False,
    }).encode()
    req = urllib.request.Request(
        'http://localhost:11434/api/chat',
        data=payload,
        headers={'Content-Type': 'application/json'}
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read())['message']['content']

print(f'Testing {len(test_cases)} prompts...\n')
passed = 0
for prompt, expected_tool, expected_kw in test_cases:
    content = ask(prompt)
    if expected_tool:
        # Expect tool_code format: tool_code print(function_name(...))
        m = re.search(r'tool_code\s+print\s*\((\w+)', content)
        actual = m.group(1) if m else None
        ok = actual == expected_tool
        status = 'PASS' if ok else 'FAIL'
        if ok: passed += 1
        print(f'[{status}] {prompt[:60]}')
        print(f'       expected={expected_tool}  got={actual}')
        if not ok:
            print(f'       raw: {content[:120]}')
    else:
        ok = expected_kw.lower() in content.lower()
        status = 'PASS' if ok else 'FAIL'
        if ok: passed += 1
        print(f'[{status}] {prompt[:60]}')
        print(f'       text response (keyword "{expected_kw}"): {"found" if ok else "MISSING"}')
    print()

print(f'Result: {passed}/{len(test_cases)} passed')

## Cell 9 — Connect Local App to vast.ai Ollama

### Step 1 — Expose Ollama via Cloudflare tunnel
In vast.ai UI → **Advanced Connection Options** → new tunnel on port **11434**.
Note the URL e.g. `https://xxxx-xxxx.trycloudflare.com`

### Step 2 — Run app locally (PowerShell)
```powershell
cd C:\Users\Aaditya\hf-space
$env:OLLAMA_BASE_URL = "https://<tunnel-url>/v1"
$env:OLLAMA_MODEL    = "aria-v2"
.venv\Scripts\python.exe application.py
```
Open browser at `http://127.0.0.1:7860`

### Keep Ollama alive after closing notebook
Run from vast.ai terminal:
```bash
OLLAMA_HOST=0.0.0.0:11434 nohup ollama serve > /tmp/ollama.log 2>&1 &
ollama list   # confirm aria-v2 is registered
```

### Pitfalls
| Problem | Fix |
|---|---|
| `connection refused` on first request | Model still loading into VRAM — wait ~30s after warmup |
| Ollama dies after notebook close | Start from terminal with `nohup`, not from notebook cell |
| `does not support tools` 400 error | Never happens with this app — it uses text parsing, not native tool_calls |
| Double BOS warning in Ollama logs | Harmless — known Gemma 4 behaviour |